# PM4Py

In [ ]:
import pm4py

In [ ]:
path = "data/example_3_ocel.json"
ocel = pm4py.read_ocel2_json(path)

In [ ]:
df_object_type_activities = pm4py.ocel_object_type_activities(ocel)
df_ocel_temporal_summary = pm4py.ocel_temporal_summary(ocel)
df_objects_summary = pm4py.ocel_objects_summary(ocel)

In [ ]:
ocdfg = pm4py.discover_ocdfg(ocel)

# View the model with the frequency annotation
pm4py.view_ocdfg(ocdfg, format="png")

In [ ]:
from pm4py.algo.transformation.ocel.graphs import object_interaction_graph

graph = object_interaction_graph.apply(ocel)

In [ ]:
model = pm4py.discover_oc_petri_net(ocel)
ocpn_view = pm4py.view_ocpn(model, format="png")

In [ ]:
df_events = ocel.events.copy()
df_events.set_index("ocel:eid", inplace=True)
df_relations = ocel.relations.copy()
df_relations.set_index("ocel:eid", inplace=True)

df_events_objects = df_events.join(df_relations, rsuffix="_relations")

In [ ]:
df_objects = ocel.objects.copy()
df_objects.set_index("ocel:oid", inplace=True)

In [ ]:
df_events.groupby("ocel:activity").describe()

# Networkx

In [ ]:
import networkx as nx

In [ ]:
_ocel_nx = pm4py.convert.convert_ocel_to_networkx(ocel)

# Workaround for https://github.com/process-intelligence-solutions/pm4py/issues/534
ocel_nx = nx.MultiDiGraph()
ocel_nx.add_nodes_from(_ocel_nx.nodes(data=True))
ocel_nx.add_edges_from([e for e in _ocel_nx.edges(data=True) if e[-1]["attr"].get("type") != "DF"])

# Add DF edges
lifecycle = (
    ocel.relations.groupby(ocel.object_id_column)
    .agg(list)
    .to_dict()[ocel.event_id_column]
)
for obj in lifecycle:
    lif = lifecycle[obj]
    for i in range(len(lif) - 1):
        ocel_nx.add_edge(
            lif[i], lif[i + 1], attr={"type": "DF", "object": obj}
        )

In [ ]:
from typing import List

def trace_backward_nx(source_event_node: str, target_activity_type: str=None, event_object_qualifiers: List[str]=[]) -> set:
    """
    source_event_node (str): source node to trace from;
    target_activity_type (str): activity type to end the trace;
    event_object_qualifiers (List[str]): list of event-object relation qualifiers used to select objects on which to trace.
    """
    nodes_to_check = [source_event_node]
    nodes_traced = set()
    while nodes_to_check:
        node = nodes_to_check.pop()
        nodes_traced.add(node)

        # End trace when target activity type is encountered
        if target_activity_type and (ocel_nx.nodes()[node]["attr"].get("ocel:activity") == target_activity_type):
            continue

        # Get objects related to event
        event_object_edges = [e for e in list(ocel_nx.out_edges(node, data=True)) if e[-1]["attr"].get("type") == "E2O"]
        trace_objects = [e[1] for e in event_object_edges]
        nodes_traced.update(trace_objects)

        # Select objects to filter DF edges on
        selected_objects = [e[1] for e in event_object_edges if e[-1]["attr"].get("qualifier", "") in event_object_qualifiers]

        trace_events = set([
            e[0] for e in list(ocel_nx.in_edges(node, data=True))
            if (e[-1]["attr"].get("type") == "DF") and (e[-1]["attr"].get("object", "") in selected_objects)
        ])

        nodes_to_check.extend(list(trace_events - nodes_traced))

    return nodes_traced


def construct_node_label(G):
    """
    Construct label based on attributes from node['attr'].
    - G: networkx graph
    """
    for node, data in list(G.nodes(data=True)):
        a = data.get("attr")
        if a is None:
            continue
        if isinstance(a, dict):
            node_type = a.get("type")
            type_activity = a.get("ocel:type") if node_type=="OBJECT" else a.get("ocel:activity")
            data["label"] = f"{node_type}_{type_activity}"


def select_node_attr(G, attr_key: str):
    """
    Add value from node['attr'][attr_key] as attribute 'selected_attr' of the nodes.
    - G: networkx graph
    """
    for node, data in list(G.nodes(data=True)):
        a = data.get("attr")
        if a is None:
            continue
        if isinstance(a, dict):
            data["selected_attr"] = a.get(attr_key)

In [ ]:
https://ysig.github.io/GraKeL/0.1a8/generated/grakel.RandomWalk.html
https://ysig.github.io/GraKeL/0.1a8/auto_examples/document_retrieval_example.html
https://ysig.github.io/GraKeL/0.1a8/auto_examples/node_attributed_dataset.html#sphx-glr-auto-examples-node-attributed-dataset-py

In [ ]:
object_types = ["PackingUnit"]

# object_types = ["HingePack"]
# activities = ["PackHinges"]

events_to_trace = df_events_objects[
    (df_events_objects["ocel:type"].isin(object_types))
].index.values

In [ ]:
trace_graphs = {}
for event in events_to_trace:
    trace_nodes = trace_backward_nx(event, "Object-creating_class_instance", ["object", "parentObject", "childObject"])
    trace_graph = nx.subgraph(ocel_nx, trace_nodes)
    construct_node_label(trace_graph)

    trace_graphs[event] = {
        "graph": trace_graph,
        "class": ocel_nx.nodes()[event]["attr"].get("averageQuality") >= 1.0,
    }

In [ ]:
from grakel.kernels import RandomWalk, SubgraphMatching, VertexHistogram, WeisfeilerLehman
from grakel.utils import graph_from_networkx

# Construct node label from attributes
for trace_graph in trace_graphs.values():
    construct_node_label(trace_graph)
    select_node_attr(trace_graph, attr_key="s_co2e[kg]")

selected_trace_graphs = {k:v for k,v in list(trace_graphs.items())[:100]}

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    edge_labels_tag="attr",
)
# gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)
# r_gk = gk.fit_transform(selected_trace_graphs_grakel)

def numeric_diff(a, b):
    try:
        return 1-(float(b)-float(a))/float(a)
    except ZeroDivisionError:
        return 0.5
    except ValueError:
        return 0.5

def dict_compare(a, b):
    sim_score = 0
    for key in a.keys():
        v_a = a.get(key)
        v_b = b.get(key)

        if not (v_a and v_b):
            sim_score += 0

        try:
            sim_score += 1-(float(v_b)-float(v_a))/float(v_a)
        except ZeroDivisionError:
            sim_score += 0.5
        except TypeError:
            sim_score += int(v_a == v_b)
        except ValueError:
            sim_score += int(v_a == v_b)
    return sim_score

sub_match = SubgraphMatching(
    normalize=True,
    kv=dict_compare,
    ke=None,
)
r_sub_match = sub_match.fit_transform(trace_graphs_grakel)

# trace_graphs_grakel = graph_from_networkx(
#     trace_graphs.values(),
#     node_labels_tag="label",
#     as_Graph=True,
# )
# random_walk = RandomWalk(n_jobs=1, normalize=True)
# r_random_walk = random_walk.fit_transform(trace_graphs_grakel)

In [ ]:
import numpy as np

k = 100 # select top k most similar graphs

target_trace_graph_id = "e_checkFemale_316>false"
target_trace_graph = trace_graphs[target_trace_graph_id]

selected_trace_graphs = {k:trace_graphs[k] for k in (list(trace_graphs.keys()))}

gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
gk.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
K_gk = gk.transform(selected_trace_graphs_grakel)

most_similar_trace_graphs_gk = np.array(list(selected_trace_graphs.keys()))[np.argsort(K_gk[:,0])[:(-1*k)]]
print(most_similar_trace_graphs_gk)

In [ ]:
import numpy as np

selected_trace_graphs = {k:trace_graphs[k] for k in most_similar_trace_graphs_gk}

sub_match = SubgraphMatching(
    normalize=True,
    kv=dict_compare,
    ke=None,
)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    edge_labels_tag="attr",
)
sub_match.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    edge_labels_tag="attr",
)
K = sub_match.transform(selected_trace_graphs_grakel)

print("Query trace")
print("--------------")
print(target_trace_graph_id)
print()
print("Most similar trace")
print("---------------------")
print(list(selected_trace_graphs.keys())[np.argsort(K[:,0])[-1]])

In [ ]:
from pandas import DataFrame

df_r = DataFrame(r_sub_match, index=selected_trace_graphs.keys(), columns=selected_trace_graphs.keys())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# pcolormesh (matplotlib) — very fast for large arrays, no text annotation
plt.figure(figsize=(12, 10))
plt.pcolormesh(df_r.values, cmap="coolwarm")
plt.colorbar(label="Normalized similarity")
plt.gca().set_xticks(np.arange(df_r.shape[1]) + 0.5)
plt.gca().set_yticks(np.arange(df_r.shape[0]) + 0.5)
plt.gca().set_xticklabels(df_r.columns, rotation=90, fontsize=8)
plt.gca().set_yticklabels(df_r.index, fontsize=8)
plt.title("Weisfeiler-Lehman subtree kernel")
plt.tight_layout()
plt.savefig("figures/plot.png")  # Save the figure

In [ ]:
import numpy as np # numpy backend
import pygmtools as pygm
import matplotlib.pyplot as plt # for plotting
from matplotlib.patches import ConnectionPatch # for plotting matching result
import networkx as nx # for plotting graphs
pygm.set_backend('numpy') # set default backend for pygmtools
np.random.seed(1) # fix random seed

In [ ]:
G1 = trace_graphs[target_trace_graph_id]
G2 = trace_graphs["e_checkFemale_3871>true"]

A1 = nx.to_numpy_array(G1)
A2 = nx.to_numpy_array(G2)

conn1, edge1 = pygm.utils.dense_to_sparse(A1)
conn2, edge2 = pygm.utils.dense_to_sparse(A2)
import functools
gaussian_aff = functools.partial(pygm.utils.gaussian_aff_fn, sigma=.1) # set affinity function
K = pygm.utils.build_aff_mat(None, edge1, conn1, None, edge2, conn2, None, None, None, None, edge_aff_fn=gaussian_aff)

X = pygm.rrwm(K, len(G1.nodes), len(G2.nodes))
X = pygm.hungarian(X)

In [ ]:
for i, g_i in enumerate(G1.nodes):
    g_j = list(G2.nodes)[np.argmax(X[i]).item()]
    a_i = G1.nodes.data()[g_i]["attr"]
    a_j = G2.nodes.data()[g_j]["attr"]

    diff = {k: (a_i[k], a_j[k]) for k in a_i if k in a_j and a_i[k] != a_j[k]}
    print(g_i, g_j, diff)

### Visualization

In [ ]:
import json, ast

def apply_node_styles_nx(G, type_attr="type", nested_attr="attr", color_map=None, default_color="#d3d3d3"):
    """
    Compute style attributes for nodes from a NetworkX graph G and store them
    as top-level node attributes so they are preserved after nx -> AGraph conversion.

    Sets: 'style' (filled), 'fillcolor' and 'tooltip' (string).
    - G: networkx.Graph or DiGraph with node data dicts
    - type_attr: key (in the node data dict or inside data['attr']) holding node type
    - nested_attr: key that may contain a dict/stringified dict with more attributes
    - color_map: dict mapping type -> color
    - default_color: fallback color
    """
    if color_map is None:
        color_map = {
            "OBJECT": "#085725",
            "EVENT":  "#009ee1",
            "ACTIVITY": "#f8ed5d",
            "DEFAULT": default_color,
        }

    for node, data in G.nodes(data=True):
        tooltip_data = {}

        # First try direct top-level type
        ntype = data.get(type_attr)

        # If nested_attr exists, try to parse it (dict or JSON / Python literal)
        nested = data.get(nested_attr)
        if nested is not None:
            if isinstance(nested, dict):
                # prefer nested type if top-level missing
                if not ntype:
                    ntype = nested.get("type") or nested.get("ocel:type")
                tooltip_data = nested

        # If still no tooltip_data, collect visible node-level attrs
        if not tooltip_data:
            # copy node data but keep only simple serializable values
            tooltip_data = {k: v for k, v in data.items() if k not in ("style", "fillcolor", "tooltip")}

        # choose color
        color = color_map.get(ntype, color_map.get("DEFAULT", default_color))

        # build tooltip string (escape double quotes)
        tooltip_items = []
        for k, v in tooltip_data.items():
            try:
                s = str(v)
            except Exception:
                s = repr(v)
            s = s.replace('"', "'")
            tooltip_items.append(f"{k}={s}")
        tooltip_str = "\n".join(tooltip_items)

        # store style attributes on the networkx node (ensure strings)
        data["style"] = "filled"
        data["fillcolor"] = str(color)
        data["tooltip"] = tooltip_str

    return G


for event, event_dict in trace_graphs.items():
    trace_graph = event_dict["graph"]
    # if event not in ["e_checkFemale_1007>true", "e_checkFemale_1017>true"]: #["e_packhinges_286", "e_packhinges_142", "e_packhinges_7"]
    #     continue
    apply_node_styles_nx(trace_graph)  # apply coloring + tooltip
    agraph = nx.nx_agraph.to_agraph(trace_graph)
    agraph.draw(f"figures/{event}.svg", prog="dot")

# apply_node_styles_nx(ocel_nx)  # apply coloring + tooltip
# agraph = nx.nx_agraph.to_agraph(ocel_nx)
# agraph.draw(f"figures/example_1_ocel.svg", prog="dot")

#### Graph alignment

In [ ]:
pos1 = nx.spring_layout(G1)
pos2 = nx.spring_layout(G2)

plt.figure(figsize=(8, 4))
ax1 = plt.subplot(1, 2, 1)
plt.title('Graph 1')
nx.draw_networkx(G1, pos1)
ax2 = plt.subplot(1, 2, 2)
plt.title('Graph 2')
nx.draw_networkx(G2, pos2)
for i, g_i in enumerate(G1.nodes):
    j = np.argmax(X[i]).item()
    g_j = list(G2.nodes)[j]
    con = ConnectionPatch(xyA=pos1[g_i], xyB=pos2[g_j], coordsA="data", coordsB="data",
                          axesA=ax1, axesB=ax2, color="green")
    plt.gca().add_artist(con)

In [ ]:
align_A2 = np.matmul(np.matmul(X, A2), X.T)
plt.figure(figsize=(8, 4))
ax1 = plt.subplot(1, 2, 1)
plt.title('Graph 1')
nx.draw_networkx(G1, pos=pos1)
ax2 = plt.subplot(1, 2, 2)
plt.title('Aligned Graph 2')
align_pos2 = {}
for i, g_i in enumerate(G1.nodes):
    j = np.argmax(X[i]).item()
    g_j = list(G2.nodes)[j]
    align_pos2[g_j] = pos1[g_i]
    con = ConnectionPatch(xyA=pos1[g_i], xyB=align_pos2[g_j], coordsA="data", coordsB="data",
                          axesA=ax1, axesB=ax2, color="green")
    plt.gca().add_artist(con)
nx.draw_networkx(G2, pos=align_pos2)

#### Heatmaps (similarity)

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list

method="average"
metric="correlation"
# compute linkage on rows/cols
Lr = linkage(df_r.values, method=method, metric=metric)
Lc = linkage(df_r.values.T, method=method, metric=metric)
ridx = leaves_list(Lr)
cidx = leaves_list(Lc)
ordered = df_r.iloc[ridx, :].iloc[:, cidx]
plt.figure(figsize=(12, 10))
plt.pcolormesh(ordered, cmap="coolwarm")
plt.colorbar(label="Normalized similarity")
plt.title("Weisfeiler-Lehman subtree kernel (clustered)")
plt.tight_layout()
plt.savefig("figures/plot.png")  # Save the figure

In [ ]:
import seaborn as sns

sns.clustermap(df_r, 
               method='average',
               metric='correlation',
               cmap='coolwarm',
               figsize=(12, 12),
               yticklabels=True,
               xticklabels=True)
plt.title('Clustered Heatmap with Dendrograms')
plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
import numpy as np

# Set up the matplotlib figure
plt.figure(figsize=(15, 5))

# Create linkage matrices
row_linkage = linkage(df_r.values, method='average', metric='correlation')
col_linkage = linkage(df_r.values.T, method='average', metric='correlation')

# Plot row dendrogram
plt.subplot(1, 2, 1)
dendrogram(row_linkage, labels=df_r.index, orientation='left')
plt.title('Row Clustering')
plt.xlabel('Distance')

# Plot column dendrogram
# plt.subplot(1, 2, 2)
# dendrogram(col_linkage, labels=df_gk.columns)
# plt.title('Column Clustering')
# plt.xlabel('Distance')
# plt.xticks(rotation=90)

plt.savefig("figures/plot.svg")
plt.tight_layout()
plt.close()

In [ ]:
from scipy.cluster.hierarchy import fcluster

def analyze_clusters(linkage_matrix, labels, max_d=None, n_clusters=None):
    """
    Analyze clusters and their labels.
    Parameters:
        linkage_matrix: scipy linkage matrix
        labels: array of labels corresponding to the clustered items
        max_d: maximum distance for clustering (alternative to n_clusters)
        n_clusters: desired number of clusters (alternative to max_d)
    Returns:
        DataFrame with cluster analysis
    """
    # Get cluster assignments
    if max_d is not None:
        clusters = fcluster(linkage_matrix, max_d, criterion='distance')
    else:
        clusters = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    
    # Create DataFrame with labels and their clusters
    df_clusters = DataFrame({
        'label': labels,
        'cluster': clusters
    })
    
    # Group by cluster and aggregate labels
    cluster_summary = (
        df_clusters
        .groupby('cluster')
        .agg(
            size=('label', 'count'),
            labels=('label', lambda x: list(x)),
        )
        .sort_values('size', ascending=False)
    )
    
    return cluster_summary

# Example usage for row clustering
n_clusters = 4  # adjust based on dendrogram inspection
cluster_summary = analyze_clusters(row_linkage, df_r.index, n_clusters=n_clusters)

# Print cluster analysis
print("\nCluster Analysis:")
for idx, row in cluster_summary.iterrows():
    print(f"\nCluster {idx} (size: {row['size']}):")
    # Print first few labels in each cluster
    print("Sample labels:", row['labels'][:5])
    
# Optional: Create a more compact visualization
plt.figure(figsize=(10, 5))
dendrogram(
    row_linkage,
    labels=df_r.index,
    orientation='left',
    leaf_rotation=0,
    leaf_font_size=8,
    truncate_mode='lastp',  # show only last p merged clusters
    p=30,  # show this many merges
    show_contracted=True,   # show collapsed sub-clusters
)
plt.title('Simplified Dendrogram with Major Clusters')
plt.tight_layout()
plt.show()